# Run GLiClass Edge and 2B locally

GLiClass v4 Edge and 2B answer decision questions about a small MIMIC-IV knowledge graph, on a laptop CPU.
- **Edge** (0.4B) is a DeBERTa encoder that reads the answer options and the text together.
- **2B** has the same decoder-KV design as 9B and 27B, on a Qwen3.5-2B backbone. Next to Edge, it separates the effect of the architecture from the effect of size.

Each question gives the model a graph excerpt as text (entities, their fields and typed relationships), followed by the decision rules.
The [Edge model card's knowledge-graph example](https://huggingface.co/knowledgator/gliclass-v4-edge#knowledge-graph-reasoning) passes the graph as text, the question as the prompt and the answers as labels. Here:
- Graph facts and rules → input text.
- Question → `prompt`.
- Candidate answers → `labels`. Each candidate has a label, such as `SOME_ADMISSIONS`, and a label description; a run shows the model one or the other.

`questions.json` has 48 questions. `question_type` says what each one tests: `lookup`, `graph`, `computation` or `semantic`.
Q23–Q30, Q42 and Q48 are twins: copies of an earlier question with edited evidence, marked by `variant_of`.

The 9B and 27B models need a GPU; see `run_9b_27b_on_pod.ipynb`.
Run this notebook from the project root, next to `src/` and `questions.json`.

## 1. Settings

Both models run on CPU in float32. Edge reads up to 2,048 tokens; 2B up to 8,192, like 9B and 27B on the pod.
Result files are named `results/<model>_<run>.json`, for example `results/2b_label_graph.json`.

In [1]:
LOCAL_MODELS = {  # result-file prefix: settings
    "edge": {
        "model_id": "knowledgator/gliclass-v4-edge",
        "device": "cpu",
        "dtype": "float32",
        "max_length": 2048,
    },
    "2b": {
        "model_id": "knowledgator/gliclass-v4-2b",
        "device": "cpu",
        "dtype": "float32",
        "max_length": 8192,
    },
}
QUESTIONS_FILE = "questions.json"

## 2. Load the questions

Each model is loaded when it runs, one at a time. The first run downloads it into the Hugging Face cache.

In [ ]:
import gc

import pandas as pd
from IPython.display import display

from src.evaluate import NO_GRAPH_ANSWER, accuracy, compare, score, twin_pairs
from src.questions import RUNS, load_questions
from src.runner import load_pipeline, run_all, run_question, save_results

questions = load_questions(QUESTIONS_FILE)
print(f"{len(questions)} questions")

## 3. Try one question

Edge's score for every candidate on one question. Label descriptions are shown here for reading only; with the default `label_source="label"` the model sees only the labels.
The elapsed time includes input processing; the first call also includes warm-up.

In [3]:
pipeline = load_pipeline(LOCAL_MODELS["edge"])
question = questions["Q21"]
result = run_question(pipeline, question)

print(question["question"])
print(f"Elapsed: {result['seconds']:.3f} seconds")
pd.DataFrame([
    {
        "label": candidate["label"],
        "description": candidate["description"],
        "score": result["scores"][candidate["label"]],
        "selected": candidate["label"] == result["selected"],
        "gold": candidate["label"] in question["expected_answer"],
    }
    for candidate in question["candidate_answers"]
]).sort_values("score", ascending=False)

Loading weights: 100%|██████████| 415/415 [00:00<00:00, 1126.66it/s]
[transformers] You are using a model of type `GLiClass` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Patient 10001217 had two admissions with intracranial abscess. Were the sets of MAIN and BASE prescription combinations for meropenem the same in both admissions or different? Compare the drug names, doses and units within each pharmacy order, counting repeated identical combinations once.
Elapsed: 0.836 seconds


,label,description,score,selected,gold
0,SAME,Both admissions have the same complete set of ...,0.560122,True,False
1,DIFFERENT,The admissions have different complete sets of...,0.358876,False,True
2,INSUFFICIENT_INFORMATION,Insufficient information to decide from the su...,0.047345,False,False
3,CONFLICTING_RECORDS,Conflicting records prevent a consistent concl...,0.033657,False,False


## 4. Run all questions

Each model answers every question seven ways (see `RUNS`):
- `label_graph`, the main run: the graph excerpt, with the labels as answer options.
- `label_nograph`, the control: the rules only, without the graph.
- `description_graph`: the graph excerpt, with the label descriptions instead of the labels, which shows how much the label wording matters.
- `label_noedges`: the graph excerpt without its relationship lines, scored against the same gold answers, which shows whether the edges are needed.
- `label_norules`: the graph excerpt without the rules, scored against the same gold answers, which shows how much the rules contribute.
- `label_nograph_norules`: the question alone, with empty text; the correct answer is INSUFFICIENT_INFORMATION.
- `description_nograph_norules`: the question alone, with the label descriptions instead of the labels; it differs from `label_nograph_norules` only in the label wording.

The models run one at a time, and each run is saved to `results/` together with the settings that produced it. 

In [4]:
def run_model(model_name):
    """Load one model, run every question seven ways (see RUNS), save each run and return them scored."""
    settings = LOCAL_MODELS[model_name]
    pipeline = load_pipeline(settings)
    run_info = {**settings, "model_name": model_name, "questions_file": QUESTIONS_FILE}
    scored = {}
    for run, options in RUNS.items():
        results = run_all(pipeline, questions, **options)
        save_results(results, f"results/{model_name}_{run}.json", **run_info, run=run, **options)
        scored[run] = score(results, questions, options["include_graph"])
        print(f"{model_name} {run}: {int(scored[run]['correct'].sum())}/{len(questions)} correct, "
              f"{results['seconds'].sum():.0f} s")
    del pipeline
    gc.collect()  # free the model's memory before the next one loads
    return scored


scored = {model_name: run_model(model_name) for model_name in LOCAL_MODELS}

display(pd.DataFrame({model_name: {run: accuracy(table).loc[0, "accuracy"] for run, table in runs.items()}
                      for model_name, runs in scored.items()}))
display(pd.DataFrame({model_name: accuracy(runs["label_graph"], by="question_type").set_index("question_type")["accuracy"]
                      for model_name, runs in scored.items()}))

Loading weights: 100%|██████████| 415/415 [00:00<00:00, 6408.75it/s]
[transformers] You are using a model of type `GLiClass` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


edge label_graph: 22/48 correct, 14 s
edge label_nograph: 20/48 correct, 6 s
edge description_graph: 19/48 correct, 14 s
edge label_noedges: 22/48 correct, 12 s
edge label_norules: 21/48 correct, 12 s
edge label_nograph_norules: 8/48 correct, 5 s
edge description_nograph_norules: 27/48 correct, 5 s


Loading weights: 100%|██████████| 682/682 [00:04<00:00, 169.99it/s]
[transformers] You are using a model of type `GLiClass` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


2b label_graph: 24/48 correct, 234 s
2b label_nograph: 29/48 correct, 222 s
2b description_graph: 31/48 correct, 274 s
2b label_noedges: 25/48 correct, 283 s
2b label_norules: 26/48 correct, 304 s
2b label_nograph_norules: 19/48 correct, 207 s
2b description_nograph_norules: 27/48 correct, 208 s


,edge,2b
label_graph,0.458333,0.500000
label_nograph,0.416667,0.604167
description_graph,0.395833,0.645833
label_noedges,0.458333,0.520833
label_norules,0.437500,0.541667
label_nograph_norules,0.166667,0.395833
description_nograph_norules,0.562500,0.562500


,edge,2b
question_type,,
computation,0.562500,0.375000
graph,0.230769,0.307692
lookup,0.400000,0.600000
semantic,0.571429,0.785714


In [5]:
with pd.option_context("display.max_rows", None):
    display(compare({model_name: runs["label_graph"] for model_name, runs in scored.items()}))

,id,question_type,twin,gold,edge_selected,edge_correct,2b_selected,2b_correct
0,Q01,computation,False,FLAG,FLAG,True,DO_NOT_FLAG,False
1,Q02,semantic,False,FAMILY_HISTORY,FAMILY_HISTORY,True,FAMILY_HISTORY,True
2,Q03,semantic,False,BOTH,BOTH,True,BOTH,True
3,Q04,semantic,False,HISTORY_ONLY,CURRENT_USE_AND_HISTORY,False,INSUFFICIENT_INFORMATION,False
4,Q05,computation,False,BEFORE,BEFORE,True,BEFORE,True
5,Q06,computation,False,INSUFFICIENT_INFORMATION,BEFORE,False,CONFLICTING_RECORDS,False
6,Q07,lookup,False,DOSE_30,DOSE_30,True,DOSE_30,True
7,Q08,lookup,False,UNIT_ML,UNIT_ML,True,UNIT_ML,True
8,Q09,computation,False,INSUFFICIENT_INFORMATION,AFTER,False,BEFORE,False
9,Q10,computation,False,YES,NO,False,NO,False


## 5. Twins

Each twin keeps its original's question, labels and rules and changes only the evidence.
`changed=False` means the model gave the same answer despite the edit.

In [6]:
for model_name, runs in scored.items():
    print(model_name)
    display(twin_pairs(runs["label_graph"]))

edge


,original,twin,original_selected,twin_selected,twin_expected,original_correct,twin_correct,changed
0,Q05,Q23,BEFORE,INSUFFICIENT_INFORMATION,INSUFFICIENT_INFORMATION,True,True,True
1,Q08,Q24,UNIT_ML,UNIT_UDCUP,INSUFFICIENT_INFORMATION,True,False,True
2,Q10,Q25,NO,INSUFFICIENT_INFORMATION,INSUFFICIENT_INFORMATION,False,True,True
3,Q15,Q26,BOTH,INFLUENZA_ONLY,INSUFFICIENT_INFORMATION,False,False,True
4,Q16,Q27,PARTIAL_DOSE,PARTIAL_DOSE,CONFLICTING_RECORDS,True,False,False
5,Q18,Q28,MATCH,MATCH,CONFLICTING_RECORDS,True,False,False
6,Q18,Q29,MATCH,MATCH,INSUFFICIENT_INFORMATION,True,False,False
7,Q22,Q30,NO,YES,INSUFFICIENT_INFORMATION,False,False,True
8,Q41,Q42,YES,YES,NO,True,False,False
9,Q17,Q48,DIFFERENT_CODES,DIFFERENT_CODES,SAME_CODES,True,False,False


2b


,original,twin,original_selected,twin_selected,twin_expected,original_correct,twin_correct,changed
0,Q05,Q23,BEFORE,AFTER,INSUFFICIENT_INFORMATION,True,False,True
1,Q08,Q24,UNIT_ML,INSUFFICIENT_INFORMATION,INSUFFICIENT_INFORMATION,True,True,True
2,Q10,Q25,NO,NO,INSUFFICIENT_INFORMATION,False,False,False
3,Q15,Q26,CONFLICTING_RECORDS,INFLUENZA_ONLY,INSUFFICIENT_INFORMATION,False,False,True
4,Q16,Q27,EXCEEDS_PRESCRIBED_DOSE,EXCEEDS_PRESCRIBED_DOSE,CONFLICTING_RECORDS,False,False,False
5,Q18,Q28,MATCH,MATCH,CONFLICTING_RECORDS,True,False,False
6,Q18,Q29,MATCH,MATCH,INSUFFICIENT_INFORMATION,True,False,False
7,Q22,Q30,NO,NO,INSUFFICIENT_INFORMATION,False,False,False
8,Q41,Q42,YES,NO,NO,True,True,True
9,Q17,Q48,DIFFERENT_CODES,DIFFERENT_CODES,SAME_CODES,True,False,False


## 6. The control: answers without the graph

In the runs without the graph (`label_nograph`, `label_nograph_norules` and `description_nograph_norules`) every question's correct answer is INSUFFICIENT_INFORMATION.
A question whose substantive gold label is still chosen is being answered from the wording or the labels, not the graph.

In [7]:
NO_GRAPH_RUNS = [run for run, options in RUNS.items() if not options["include_graph"]]
for model_name, runs in scored.items():
    for run in NO_GRAPH_RUNS:
        no_graph = runs[run]
        answered_without_graph = (no_graph["selected"] == no_graph["gold"]) & (no_graph["gold"] != NO_GRAPH_ANSWER)
        print(f"{model_name} {run}: abstained on {int(no_graph['correct'].sum())}/{len(no_graph)}; "
              "substantive gold label chosen without the graph:")
        display(no_graph.loc[answered_without_graph, ["id", "question_type", "gold"]])

edge label_nograph: abstained on 20/48; substantive gold label chosen without the graph:


,id,question_type,gold
1,Q02,semantic,FAMILY_HISTORY
2,Q03,semantic,BOTH
9,Q10,computation,YES
17,Q18,graph,MATCH
30,Q31,semantic,NOT_ALCOHOL_RELATED
36,Q37,semantic,BOTH
39,Q40,semantic,HISTORY_ONLY
41,Q42,computation,NO


edge label_nograph_norules: abstained on 8/48; substantive gold label chosen without the graph:


,id,question_type,gold
0,Q01,computation,FLAG
2,Q03,semantic,BOTH
7,Q08,lookup,UNIT_ML
9,Q10,computation,YES
16,Q17,graph,DIFFERENT_CODES
17,Q18,graph,MATCH
20,Q21,graph,DIFFERENT
31,Q32,semantic,BOTH
34,Q35,semantic,PERSONAL_ONLY
36,Q37,semantic,BOTH


edge description_nograph_norules: abstained on 27/48; substantive gold label chosen without the graph:


,id,question_type,gold
9,Q10,computation,YES
14,Q15,lookup,INFLUENZA_ONLY
19,Q20,graph,SOME_ADMISSIONS
21,Q22,graph,YES
41,Q42,computation,NO
47,Q48,graph,SAME_CODES


... (2 outputs left out)

2b label_nograph_norules: abstained on 19/48; substantive gold label chosen without the graph:


,id,question_type,gold
9,Q10,computation,YES
17,Q18,graph,MATCH
35,Q36,semantic,YES
41,Q42,computation,NO
42,Q43,computation,YES
45,Q46,semantic,SUPPORTED
46,Q47,computation,NOT_WITHIN_30_DAYS
47,Q48,graph,SAME_CODES


2b description_nograph_norules: abstained on 27/48; substantive gold label chosen without the graph:


,id,question_type,gold
0,Q01,computation,FLAG
14,Q15,lookup,INFLUENZA_ONLY
17,Q18,graph,MATCH
21,Q22,graph,YES
45,Q46,semantic,SUPPORTED
47,Q48,graph,SAME_CODES


In [8]:
for model_name, runs in scored.items():
    print(model_name)
    with pd.option_context("display.max_rows", None):
        display(compare(runs))

edge


,id,question_type,twin,gold,label_graph_selected,label_graph_correct,label_nograph_selected,label_nograph_correct,description_graph_selected,description_graph_correct,label_noedges_selected,label_noedges_correct,label_norules_selected,label_norules_correct,label_nograph_norules_selected,label_nograph_norules_correct,description_nograph_norules_selected,description_nograph_norules_correct
0,Q01,computation,False,FLAG,FLAG,True,DO_NOT_FLAG,False,DO_NOT_FLAG,False,FLAG,True,FLAG,True,FLAG,False,DO_NOT_FLAG,False
1,Q02,semantic,False,FAMILY_HISTORY,FAMILY_HISTORY,True,FAMILY_HISTORY,False,FAMILY_HISTORY,True,FAMILY_HISTORY,True,FAMILY_HISTORY,True,PATIENT_DIAGNOSIS,False,INSUFFICIENT_INFORMATION,True
2,Q03,semantic,False,BOTH,BOTH,True,BOTH,False,DEPRESSION_ONLY,False,NEITHER_RECORDED,False,BOTH,True,BOTH,False,INSUFFICIENT_INFORMATION,True
3,Q04,semantic,False,HISTORY_ONLY,CURRENT_USE_AND_HISTORY,False,CURRENT_USE_AND_HISTORY,False,INSUFFICIENT_INFORMATION,False,HISTORY_ONLY,True,CURRENT_USE_AND_HISTORY,False,CURRENT_USE_AND_HISTORY,False,INSUFFICIENT_INFORMATION,True
4,Q05,computation,False,BEFORE,BEFORE,True,INSUFFICIENT_INFORMATION,True,BEFORE,True,BEFORE,True,BEFORE,True,AFTER,False,INSUFFICIENT_INFORMATION,True
5,Q06,computation,False,INSUFFICIENT_INFORMATION,BEFORE,False,INSUFFICIENT_INFORMATION,True,BEFORE,False,BEFORE,False,BEFORE,False,AFTER,False,INSUFFICIENT_INFORMATION,True
6,Q07,lookup,False,DOSE_30,DOSE_30,True,INSUFFICIENT_INFORMATION,True,DOSE_30,True,DOSE_30,True,DOSE_30,True,INSUFFICIENT_INFORMATION,True,INSUFFICIENT_INFORMATION,True
7,Q08,lookup,False,UNIT_ML,UNIT_ML,True,INSUFFICIENT_INFORMATION,True,UNIT_ML,True,UNIT_ML,True,UNIT_ML,True,UNIT_ML,False,INSUFFICIENT_INFORMATION,True
8,Q09,computation,False,INSUFFICIENT_INFORMATION,AFTER,False,INSUFFICIENT_INFORMATION,True,AFTER,False,AFTER,False,AFTER,False,AFTER,False,INSUFFICIENT_INFORMATION,True
9,Q10,computation,False,YES,NO,False,YES,False,NO,False,NO,False,NO,False,YES,False,YES,False


2b


,id,question_type,twin,gold,label_graph_selected,label_graph_correct,label_nograph_selected,label_nograph_correct,description_graph_selected,description_graph_correct,label_noedges_selected,label_noedges_correct,label_norules_selected,label_norules_correct,label_nograph_norules_selected,label_nograph_norules_correct,description_nograph_norules_selected,description_nograph_norules_correct
0,Q01,computation,False,FLAG,DO_NOT_FLAG,False,DO_NOT_FLAG,False,FLAG,True,DO_NOT_FLAG,False,DO_NOT_FLAG,False,DO_NOT_FLAG,False,FLAG,False
1,Q02,semantic,False,FAMILY_HISTORY,FAMILY_HISTORY,True,PATIENT_DIAGNOSIS,False,FAMILY_HISTORY,True,FAMILY_HISTORY,True,FAMILY_HISTORY,True,PATIENT_DIAGNOSIS,False,PATIENT_DIAGNOSIS,False
2,Q03,semantic,False,BOTH,BOTH,True,INSUFFICIENT_INFORMATION,True,BOTH,True,BOTH,True,BOTH,True,INSUFFICIENT_INFORMATION,True,INSUFFICIENT_INFORMATION,True
3,Q04,semantic,False,HISTORY_ONLY,INSUFFICIENT_INFORMATION,False,INSUFFICIENT_INFORMATION,True,HISTORY_ONLY,True,INSUFFICIENT_INFORMATION,False,INSUFFICIENT_INFORMATION,False,INSUFFICIENT_INFORMATION,True,INSUFFICIENT_INFORMATION,True
4,Q05,computation,False,BEFORE,BEFORE,True,INSUFFICIENT_INFORMATION,True,BEFORE,True,BEFORE,True,BEFORE,True,INSUFFICIENT_INFORMATION,True,INSUFFICIENT_INFORMATION,True
5,Q06,computation,False,INSUFFICIENT_INFORMATION,CONFLICTING_RECORDS,False,INSUFFICIENT_INFORMATION,True,INSUFFICIENT_INFORMATION,True,CONFLICTING_RECORDS,False,CONFLICTING_RECORDS,False,AFTER,False,BEFORE,False
6,Q07,lookup,False,DOSE_30,DOSE_30,True,INSUFFICIENT_INFORMATION,True,DOSE_30,True,DOSE_30,True,DOSE_30,True,INSUFFICIENT_INFORMATION,True,INSUFFICIENT_INFORMATION,True
7,Q08,lookup,False,UNIT_ML,UNIT_ML,True,INSUFFICIENT_INFORMATION,True,UNIT_ML,True,UNIT_ML,True,UNIT_ML,True,UNIT_MG,False,INSUFFICIENT_INFORMATION,True
8,Q09,computation,False,INSUFFICIENT_INFORMATION,BEFORE,False,INSUFFICIENT_INFORMATION,True,BEFORE,False,BEFORE,False,BEFORE,False,BEFORE,False,BEFORE,False
9,Q10,computation,False,YES,NO,False,NO,False,YES,True,NO,False,YES,True,YES,False,INSUFFICIENT_INFORMATION,True


In [9]:
rules_df = pd.read_csv("questions_rules.csv")
with pd.option_context("display.max_colwidth", None):
    display(rules_df)

,question_id,question,rules
0,Q01,Should the admission 27703517 for the patient be flagged as a return within 0 through 30 calendar days inclusive after discharge from the admission 24597018?,"R01, R07"
1,Q02,"In the admission with hadm_id 24597018, does the diagnosis with icd_code V161 document lung cancer in the patient, or a family history of that cancer?","R02, R10"
2,Q03,"Does the admission with hadm_id 25563031 record depression, anxiety, both, or neither of those diagnoses?",R10
3,Q04,"In the admission with hadm_id 22595853, is the patient with id 10000032 coded as a current smoker or a former smoker?",R10
4,Q05,"During the admission 24597018, was the spinal tap recorded before or after the procedure described as Other incision of brain?","R01, R07"
5,Q06,"On 2141-05-22, in admission 24982426, was left-heart sampling and pressure measurement performed before or after coronary-artery fluoroscopy?","R01, R07"
6,Q07,"For the lactulose pharmacy order with id 13724136 given in admission 22841357, what numeric amount was prescribed per dose?","R01, R04"
7,Q08,"For the lactulose order with pharmacy_id 13724136 in the admission 22841357, what unit was recorded for the prescribed dose?","R01, R04"
8,Q09,"In the admission 24597018, did the earliest meropenem prescription start before or after the brain procedure?","R01, R07"
9,Q10,"Does admission with hadm_id 25742920 fall within 0 through 30 calendar days inclusive of the most recent preceding discharge for the patient 10000032? Compare discharge and admission dates, not elapsed hours.","R01, R07"
